_Manuscript: Methods — Feature selection._

# Feature Selection with Leakage-Free Cross-Validation

This notebook performs feature selection **inside** the cross-validation loop so that
the reported accuracy is not optimistically biased. Feature selection is performed within each cross-validation fold so that reported
performance is not optimistically biased.

**Design**
1. All preprocessing + selection is wrapped in a single `Pipeline`.
2. The pipeline is evaluated with `cross_validate`, so the correlation filter and RFE are
   **re-fit on each training fold only** — the held-out fold never influences which features are chosen.
3. Optionally, an outer **spatial block CV** replaces the random KFold to also account for
   spatial autocorrelation.
4. A final selection is run **once on all data** purely to fix the band list for the deployed
   per-year maps — reported metrics come from the cross-validation above.

Set `CSV_PATH` to your real training table and run top to bottom.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.inspection import permutation_importance
from sklearn.model_selection import (StratifiedKFold, cross_validate,
                                     cross_val_predict, GroupKFold)
from sklearn.metrics import (f1_score, cohen_kappa_score, make_scorer,
                             classification_report, confusion_matrix)

# ============================ USER INPUTS ============================
CSV_PATH       = Path(r"DATA_DIR\test_small\Case_5\training_data_cleaned.csv")
LABEL_COL      = "label"          # name of the class column
N_FEATURES     = 30               # final feature-set size to evaluate
CORR_THRESHOLD = 0.95             # Pearson |r| above which a feature is dropped
N_SPLITS       = 5
RANDOM_STATE   = 42
RF_SELECT      = dict(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)  # cheap RF used inside RFE
RF_FINAL       = dict(n_estimators=500, random_state=RANDOM_STATE, n_jobs=-1)  # the classifier you report
# Optional: column with a spatial block ID per sample for spatial CV (else leave None)
BLOCK_COL      = None             # e.g. "block_id"
# ====================================================================

In [ ]:
df = pd.read_csv(CSV_PATH)
y = df[LABEL_COL].values
feature_cols = [c for c in df.columns if c not in (LABEL_COL, BLOCK_COL)]
X = df[feature_cols].values
groups = df[BLOCK_COL].values if BLOCK_COL else None
print(f"{X.shape[0]} samples, {X.shape[1]} candidate features, classes: {np.unique(y)}")

## 1. Correlation filter as a CV-safe transformer

Wrapping the Pearson |r| > threshold drop in a transformer means it is **re-fit on the
training rows of each fold**, instead of once on the whole table. Which features get dropped
can differ slightly fold to fold — that is correct, because the decision is now made without
seeing the held-out data.

In [ ]:
class CorrelationFilter(BaseEstimator, TransformerMixin):
    # Drop one of each pair of features with |Pearson r| > threshold.
    # Fit on training data only; kept-column indices learned in fit().
    def __init__(self, threshold=0.95):
        self.threshold = threshold

    def fit(self, X, y=None):
        Xdf = pd.DataFrame(X)
        corr = Xdf.corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        drop = [c for c in upper.columns if (upper[c] > self.threshold).any()]
        self.keep_idx_ = [c for c in range(Xdf.shape[1]) if c not in drop]
        return self

    def transform(self, X):
        return np.asarray(X)[:, self.keep_idx_]

## 2. The pipeline

`StandardScaler → CorrelationFilter → RFE → RandomForest`. Everything that *learns anything
from the data* is inside this object, so `cross_validate` re-fits all of it per fold.

In [ ]:
def make_pipeline(n_features=N_FEATURES):
    return Pipeline([
        ("scale", StandardScaler()),
        ("corr",  CorrelationFilter(CORR_THRESHOLD)),
        ("rfe",   RFE(RandomForestClassifier(**RF_SELECT),
                      n_features_to_select=n_features)),
        ("rf",    RandomForestClassifier(**RF_FINAL)),
    ])

## 3. Leakage-free cross-validation (the numbers you report)

`StratifiedKFold` for a random split, or `GroupKFold` on your spatial block IDs to also
satisfy the spatial-independence comments. The F1 / Kappa printed here are the honest
estimates that replace the old ones.

In [ ]:
scoring = {
    "f1_weighted": make_scorer(f1_score, average="weighted"),
    "kappa":       make_scorer(cohen_kappa_score),
    "accuracy":    "accuracy",
}

if BLOCK_COL:
    cv = GroupKFold(n_splits=N_SPLITS)
    split_kwargs = dict(groups=groups)
    print(f"Using GroupKFold (spatial blocks) with {N_SPLITS} folds")
else:
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    split_kwargs = {}
    print(f"Using StratifiedKFold with {N_SPLITS} folds")

cv_res = cross_validate(make_pipeline(), X, y, cv=cv, scoring=scoring,
                        return_train_score=False, n_jobs=-1, **split_kwargs)

for m in ["f1_weighted", "kappa", "accuracy"]:
    s = cv_res[f"test_{m}"]
    print(f"{m:12s}: {s.mean():.3f} ± {s.std():.3f}")

## 4. Confusion matrix from out-of-fold predictions

`cross_val_predict` gives a prediction for every sample made by a model that did **not**
see it during training or selection — a clean basis for the confusion matrix / per-class report.

In [ ]:
y_oof = cross_val_predict(make_pipeline(), X, y, cv=cv, n_jobs=-1, **split_kwargs)
print(confusion_matrix(y, y_oof))
print()
print(classification_report(y, y_oof, digits=3))

## 5. Sensitivity to feature-set size (top-20 vs top-30)

Re-run the leakage-free CV at each size and compare. This is the honest version of your
"I tested 20 and 30 and chose 30" decision — now the choice is made on un-inflated numbers.

In [ ]:
# Note: a size larger than the number of features surviving the correlation
# filter on a given fold will make RFE fail, so cap the grid accordingly.
for k in [15, 20, 25, 30, 35]:
    r = cross_validate(make_pipeline(k), X, y, cv=cv,
                       scoring={"f1": make_scorer(f1_score, average="weighted")},
                       n_jobs=-1, error_score=np.nan, **split_kwargs)
    f1m = r['test_f1'].mean()
    flag = "  (NaN -> size exceeds features kept after corr-filter)" if np.isnan(f1m) else ""
    print(f"n_features={k:2d}  F1={f1m:.3f} ± {r['test_f1'].std():.3f}{flag}")

## 6. Final feature list for the deployed maps

Selecting the *deployed* band set requires fitting selection once on all data — you must
commit to one list to generate the yearly rasters. This is legitimate **as long as the
reported accuracy comes from Section 3, not from re-testing on this same selection.**
The permutation importances below are computed for the appendix figure; note they describe
the fitted model, they are not a second selection step.

In [ ]:
final_pipe = make_pipeline().fit(X, y)

# Recover which original features survived corr-filter -> RFE
kept_after_corr = np.array(feature_cols)[final_pipe.named_steps["corr"].keep_idx_]
rfe_support     = final_pipe.named_steps["rfe"].support_
selected_features = kept_after_corr[rfe_support]
print(f"{len(selected_features)} features selected for deployment:")
for f in selected_features:
    print("  ", f)

In [ ]:
# Permutation importance on the selected features (for Appendix figure only).
# Computed on the full fitted model; for a stricter version, compute on a held-out split.
X_sel = df[list(selected_features)].values
imp_model = RandomForestClassifier(**RF_FINAL).fit(StandardScaler().fit_transform(X_sel), y)
perm = permutation_importance(imp_model, StandardScaler().fit_transform(X_sel), y,
                              n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
imp = (pd.Series(perm.importances_mean, index=selected_features)
       .sort_values(ascending=False))
print(imp)

In [ ]:
# Save the deployable feature list so the per-year raster step can read it back
out = Path(CSV_PATH).with_name("selected_features_final.csv")
pd.DataFrame({"feature": selected_features}).to_csv(out, index=False)
print("Saved ->", out)